# Customer Churn Analysis

**Objective:** Identify patterns behind subscription cancellations by examining customer activity, engagement, service usage, billing, and customer segments. Build a baseline model and propose retention improvements.

**How to use:** Open this notebook in Google Colab, run cells from top to bottom, and upload your customer CSV when prompted. The notebook is designed for common churn datasets (including Telco Customer Churn). If your target column has a different name, edit `TARGET` in Section 3.

> Results show associations, not proof of causation. Validate retention ideas with controlled experiments.


## 1. Import libraries and upload data

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Understand and clean the dataset

In [ ]:
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

# Common Telco dataset column may be stored as text.
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Columns:", df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("data_type"))
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)


## 3. Define churn target and calculate churn rate

In [ ]:
# Change this if your dataset uses another target-column name.
TARGET = "Churn"

if TARGET not in df.columns:
    matches = [c for c in df.columns if c.lower() in
               ["churn", "exited", "cancelled", "canceled", "is_churned"]]
    if not matches:
        raise ValueError("Set TARGET to the column that identifies customers who churned.")
    TARGET = matches[0]

print("Target column:", TARGET)
print("Target values:", df[TARGET].value_counts(dropna=False).to_dict())

# Common positive labels. Add your dataset's churn label here if necessary.
positive_values = {"yes", "true", "1", "churned", "left", "cancelled", "canceled"}
df["_churn_flag"] = df[TARGET].astype(str).str.strip().str.lower().isin(positive_values).astype(int)

if df["_churn_flag"].nunique() != 2:
    raise ValueError("Target must contain both churn and retained customers. Check positive_values.")

churn_rate = 100 * df["_churn_flag"].mean()
print(f"Overall churn rate: {churn_rate:.2f}%")
print("Customers:", len(df))
print("Churned customers:", int(df["_churn_flag"].sum()))


## 4. Churn distribution

In [ ]:
df[TARGET].value_counts().plot(kind="bar", figsize=(6,4))
plt.title("Customer Churn Distribution")
plt.xlabel("Churn status")
plt.ylabel("Number of customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Activity, usage, and engagement analysis

The notebook compares numeric measures between retained and churned customers. Common examples include tenure, monthly charges, total charges, logins, sessions, usage, and support tickets.

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=np.number).columns
                if c != "_churn_flag"]

if numeric_cols:
    comparison = df.groupby("_churn_flag")[numeric_cols].mean().T
    comparison.columns = ["Retained", "Churned"]
    comparison["Difference (churned - retained)"] = comparison["Churned"] - comparison["Retained"]
    display(comparison.sort_values("Difference (churned - retained)", key=abs, ascending=False))

    preferred = ["tenure", "MonthlyCharges", "TotalCharges", "Usage", "Logins",
                 "Sessions", "SupportTickets", "LastLoginDays"]
    plot_cols = [c for c in preferred if c in numeric_cols] or numeric_cols[:4]
    for col in plot_cols:
        df.boxplot(column=col, by="_churn_flag", figsize=(6,4))
        plt.title(f"{col} by churn status")
        plt.suptitle("")
        plt.xlabel("0 = retained, 1 = churned")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric engagement or activity columns found.")


## 6. Segment-level churn analysis

This checks common customer segments such as contract type, payment method, service type, plan, region, and customer profile.

In [ ]:
candidate_segments = [
    "Contract", "PaymentMethod", "InternetService", "SeniorCitizen",
    "gender", "Partner", "Dependents", "PaperlessBilling",
    "DeviceType", "Plan", "Region"
]
segment_cols = [c for c in candidate_segments if c in df.columns]
if not segment_cols:
    segment_cols = [c for c in df.select_dtypes(include="object").columns
                    if c != TARGET and df[c].nunique() <= 12][:6]

for col in segment_cols:
    rates = 100 * df.groupby(col, dropna=False)["_churn_flag"].mean()
    print(f"Churn rate by {col} (%)")
    display(rates.sort_values(ascending=False).round(2).to_frame("churn_rate_percent"))
    rates.sort_values(ascending=False).plot(kind="bar", figsize=(8,4))
    plt.title(f"Churn Rate by {col}")
    plt.ylabel("Churn rate (%)")
    plt.xlabel(col)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


## 7. Numeric correlations with churn

In [ ]:
corr_cols = numeric_cols + ["_churn_flag"]
corr = df[corr_cols].corr(numeric_only=True)
display(corr[["_churn_flag"]].sort_values("_churn_flag", ascending=False))
plt.figure(figsize=(8,6))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Numeric Feature Correlation Matrix")
plt.tight_layout()
plt.show()


## 8. Train a baseline churn prediction model

A Random Forest classifier provides a starting point for identifying customers with elevated churn risk. It should not be treated as a definitive prediction.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score

drop_cols = [TARGET, "_churn_flag"]
drop_cols += [c for c in df.columns if c.lower() in
              ["customerid", "customer_id", "userid", "user_id", "accountid", "account_id"]]
X = df.drop(columns=drop_cols, errors="ignore")
y = df["_churn_flag"]

num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_features),
    ("cat", categorical_pipe, cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced", min_samples_leaf=2
    ))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Churn Model Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Most influential model features

In [ ]:
feature_names = model.named_steps["preprocess"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False).head(15)
display(importance_df)

importance_df.sort_values("importance").plot(
    x="feature", y="importance", kind="barh", legend=False, figsize=(8,6)
)
plt.title("Top 15 Predictive Features")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()
print("Feature importance indicates predictive usefulness, not causation.")


## 10. Review customers with higher predicted risk

In [ ]:
risk_view = df.loc[X_test.index].copy()
risk_view["predicted_churn_probability"] = prob
risk_view["risk_band"] = pd.cut(
    prob, bins=[-0.001, 0.30, 0.60, 1.001],
    labels=["Low", "Medium", "High"]
)
display_cols = [c for c in [
    "customerID", "customer_id", "Contract", "tenure",
    "MonthlyCharges", "PaymentMethod"
] if c in risk_view.columns]
display_cols += ["predicted_churn_probability", "risk_band"]
display(risk_view.sort_values(
    "predicted_churn_probability", ascending=False
)[display_cols].head(20))


## 11. Retention improvement recommendations

Use the dataset findings to prioritize and test suitable actions:

1. **Low engagement:** Provide onboarding guidance, feature education, and relevant usage reminders.
2. **Early-life churn:** Improve the first 30–90 days with setup assistance and proactive check-ins.
3. **Price or billing concerns:** Make bills easier to understand and offer plans that better match customer needs.
4. **Repeated support issues:** Track repeat complaints, improve resolution time, and follow up after support interactions.
5. **Contract or plan risk:** Test transparent loyalty benefits or flexible plan options; avoid making cancellation difficult.
6. **Declining activity:** Trigger helpful outreach when logins, sessions, or product usage fall noticeably.

**Measure impact:** Run controlled retention experiments and compare churn, satisfaction, and revenue against a control group. Target interventions to the observed problem rather than giving every customer the same discount.


## 12. Final project conclusion

Complete this after running the analysis:

- Overall churn rate: **[insert percentage]**
- Highest-churn segment: **[insert segment]**
- Most important activity/engagement signal: **[insert feature]**
- Other notable churn factors: **[insert factors]**
- Model ROC-AUC: **[insert score]**
- Recommended retention action: **[insert action]**

**Conclusion:** This analysis identifies customer segments and behavioral patterns associated with subscription cancellation. The business can use these findings to improve onboarding, engagement, service quality, and plan fit, then measure the effectiveness of retention initiatives.
